# MLP para classificação de diabetes

Notebook autocontido. Toda a implementação está neste arquivo, organizada em funções pequenas, explícitas e executáveis em ordem.

## 1. Configuração

Valores do experimento e validações independentes.

In [ ]:
"""Configuração explícita do experimento."""

import os
from pathlib import Path


class ExperimentConfig:
    def __init__(self, project_root: Path, data_path: Path, artifacts_directory: Path, seed: int, training_fraction: float, validation_fraction: float, test_fraction: float, batch_size: int, epochs: int, minimum_epochs: int, threshold_minimum: float, threshold_maximum: float, threshold_step: float, learning_rate: float, weight_decay: float, hidden_dimensions: list[int], dropout: float, output_size: int, early_stopping_patience: int, log_interval: int, requested_device: str | None, maximum_rows: int | None, positive_weight_multiplier: float) -> None:
        self.project_root, self.data_path, self.artifacts_directory = project_root, data_path, artifacts_directory
        self.seed, self.training_fraction, self.validation_fraction, self.test_fraction = seed, training_fraction, validation_fraction, test_fraction
        self.batch_size, self.epochs, self.minimum_epochs = batch_size, epochs, minimum_epochs
        self.threshold_minimum, self.threshold_maximum, self.threshold_step = threshold_minimum, threshold_maximum, threshold_step
        self.learning_rate, self.weight_decay = learning_rate, weight_decay
        self.hidden_dimensions, self.dropout, self.output_size = hidden_dimensions, dropout, output_size
        self.early_stopping_patience, self.log_interval = early_stopping_patience, log_interval
        self.requested_device, self.maximum_rows = requested_device, maximum_rows
        self.positive_weight_multiplier = positive_weight_multiplier


def create_default_config(project_root: Path) -> ExperimentConfig:
    return ExperimentConfig(project_root, project_root / "dataset" / "diabetes_prediction_dataset.csv", project_root / "artifacts" / "f1_full_dataset", 42, .70, .15, .15, 32, 60, 30, .05, .95, .01, 1e-3, 1e-4, [32, 16], .30, 1, 8, 1, None, None, 1.0)


def read_optional_integer_environment_variable(name: str) -> int | None:
    value = os.getenv(name)
    return None if value in (None, "") else int(value)


def apply_environment_overrides(config: ExperimentConfig) -> ExperimentConfig:
    artifacts_value = os.getenv("MLP_ARTIFACTS_DIR")
    if artifacts_value: config.artifacts_directory = Path(artifacts_value)
    for name, attribute in [("MLP_EPOCHS", "epochs"), ("MLP_BATCH_SIZE", "batch_size"), ("MLP_MAX_ROWS", "maximum_rows")]:
        value = read_optional_integer_environment_variable(name)
        if value is not None: setattr(config, attribute, value)
    device_value = os.getenv("MLP_DEVICE")
    if device_value is not None: config.requested_device = device_value or None
    multiplier_value = os.getenv("MLP_POSITIVE_WEIGHT_MULTIPLIER")
    if multiplier_value is not None: config.positive_weight_multiplier = float(multiplier_value)
    return config


def validate_config(config: ExperimentConfig) -> None:
    if abs(config.training_fraction + config.validation_fraction + config.test_fraction - 1.0) > 1e-9: raise ValueError("As frações devem somar 1.0.")
    if min(config.training_fraction, config.validation_fraction, config.test_fraction, config.learning_rate, config.threshold_minimum, config.threshold_step) <= 0: raise ValueError("Os valores de treino e limiar devem ser positivos.")
    if config.threshold_maximum >= 1 or config.threshold_minimum >= config.threshold_maximum: raise ValueError("Os limites de limiar são inválidos.")
    if config.positive_weight_multiplier < 0: raise ValueError("positive_weight_multiplier não pode ser negativo.")
    if config.batch_size <= 0 or config.epochs <= 0 or config.minimum_epochs <= 0 or config.minimum_epochs > config.epochs: raise ValueError("Valores de época ou lote inválidos.")
    if config.weight_decay < 0 or config.early_stopping_patience <= 0 or config.log_interval <= 0: raise ValueError("Valores de otimização inválidos.")
    if config.maximum_rows is not None and config.maximum_rows <= 0: raise ValueError("O limite de linhas deve ser positivo.")
    if config.output_size != 1 or not 0 <= config.dropout < 1 or not config.hidden_dimensions or min(config.hidden_dimensions) <= 0: raise ValueError("Configuração do modelo inválida.")


def get_checkpoint_path(config: ExperimentConfig) -> Path: return config.artifacts_directory / "best_model.pt"
def get_preprocessor_path(config: ExperimentConfig) -> Path: return config.artifacts_directory / "preprocessor.joblib"

def config_to_dictionary(config: ExperimentConfig) -> dict[str, object]:
    return {"project_root": str(config.project_root), "data_path": str(config.data_path), "artifacts_directory": str(config.artifacts_directory), "seed": config.seed, "training_fraction": config.training_fraction, "validation_fraction": config.validation_fraction, "test_fraction": config.test_fraction, "batch_size": config.batch_size, "epochs": config.epochs, "minimum_epochs": config.minimum_epochs, "threshold_minimum": config.threshold_minimum, "threshold_maximum": config.threshold_maximum, "threshold_step": config.threshold_step, "learning_rate": config.learning_rate, "weight_decay": config.weight_decay, "hidden_dimensions": list(config.hidden_dimensions), "dropout": config.dropout, "output_size": config.output_size, "early_stopping_patience": config.early_stopping_patience, "log_interval": config.log_interval, "requested_device": config.requested_device, "maximum_rows": config.maximum_rows, "positive_weight_multiplier": config.positive_weight_multiplier}

## 2. Ambiente e reprodutibilidade

Sementes, determinismo e seleção explícita de CPU ou CUDA.

In [ ]:
"""Seleção de dispositivo e reprodutibilidade."""

import platform
import random
import sys

import numpy as np
import torch


class RuntimeMetadata:
    def __init__(
        self,
        python_version: str,
        platform_name: str,
        torch_version: str,
        torch_cuda_version: str | None,
        device_name: str,
        cuda_available: bool,
        gpu_name: str | None,
        cuda_device_count: int | None,
    ) -> None:
        self.python_version = python_version
        self.platform_name = platform_name
        self.torch_version = torch_version
        self.torch_cuda_version = torch_cuda_version
        self.device_name = device_name
        self.cuda_available = cuda_available
        self.gpu_name = gpu_name
        self.cuda_device_count = cuda_device_count

    def to_dictionary(self) -> dict[str, object]:
        values: dict[str, object] = {}
        values["python"] = self.python_version
        values["platform"] = self.platform_name
        values["torch"] = self.torch_version
        values["torch_cuda_version"] = self.torch_cuda_version
        values["device"] = self.device_name
        values["cuda_available"] = self.cuda_available
        if self.gpu_name is not None:
            values["gpu_name"] = self.gpu_name
        if self.cuda_device_count is not None:
            values["cuda_device_count"] = self.cuda_device_count
        return values


def seed_python(seed: int) -> None:
    random.seed(seed)


def seed_numpy(seed: int) -> None:
    np.random.seed(seed)


def seed_torch(seed: int) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def configure_deterministic_torch() -> None:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def configure_reproducibility(seed: int) -> None:
    seed_python(seed)
    seed_numpy(seed)
    seed_torch(seed)
    configure_deterministic_torch()


def validate_requested_device(device_name: str) -> None:
    requested_device = torch.device(device_name)
    if requested_device.type == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA foi solicitada, mas não está disponível.")


def select_device(device_name: str | None) -> torch.device:
    if device_name is not None:
        validate_requested_device(device_name)
        return torch.device(device_name)

    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def collect_runtime_metadata(device: torch.device) -> RuntimeMetadata:
    gpu_name = None
    cuda_device_count = None
    if device.type == "cuda":
        gpu_name = torch.cuda.get_device_name(device)
        cuda_device_count = torch.cuda.device_count()

    metadata = RuntimeMetadata(
        python_version=sys.version,
        platform_name=platform.platform(),
        torch_version=torch.__version__,
        torch_cuda_version=torch.version.cuda,
        device_name=str(device),
        cuda_available=torch.cuda.is_available(),
        gpu_name=gpu_name,
        cuda_device_count=cuda_device_count,
    )
    return metadata


def print_device_summary(metadata: RuntimeMetadata) -> None:
    print("Dispositivo selecionado: " + metadata.device_name)
    if metadata.gpu_name is not None:
        cuda_version = str(metadata.torch_cuda_version)
        print("GPU: " + metadata.gpu_name + " | CUDA PyTorch: " + cuda_version)
        return
    print("CUDA indisponível ou não solicitado; usando CPU.")



## 3. Leitura e divisão dos dados

Cada regra de validação do CSV possui uma função própria.

In [ ]:
"""Leitura, validação e divisão do dataset."""

from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split



CATEGORICAL_COLUMNS = ["gender", "smoking_history"]
NUMERIC_COLUMNS = [
    "age",
    "hypertension",
    "heart_disease",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level",
]
TARGET_COLUMN = "diabetes"
FEATURE_COLUMNS = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS
REQUIRED_COLUMNS = FEATURE_COLUMNS + [TARGET_COLUMN]


class FeaturesAndTarget:
    def __init__(self, features: pd.DataFrame, target: pd.Series) -> None:
        self.features = features
        self.target = target


class RemainingAndTestData:
    def __init__(
        self,
        remaining_features: pd.DataFrame,
        test_features: pd.DataFrame,
        remaining_target: pd.Series,
        test_target: pd.Series,
    ) -> None:
        self.remaining_features = remaining_features
        self.test_features = test_features
        self.remaining_target = remaining_target
        self.test_target = test_target


class DatasetSplits:
    def __init__(
        self,
        x_train: pd.DataFrame,
        x_validation: pd.DataFrame,
        x_test: pd.DataFrame,
        y_train: pd.Series,
        y_validation: pd.Series,
        y_test: pd.Series,
    ) -> None:
        self.x_train = x_train
        self.x_validation = x_validation
        self.x_test = x_test
        self.y_train = y_train
        self.y_validation = y_validation
        self.y_test = y_test


class TargetSummary:
    def __init__(self, row_count: int, positive_rate: float, positive_count: int) -> None:
        self.row_count = row_count
        self.positive_rate = positive_rate
        self.positive_count = positive_count

    def to_dictionary(self) -> dict[str, float | int]:
        values: dict[str, float | int] = {}
        values["rows"] = self.row_count
        values["positive_rate"] = self.positive_rate
        values["positive_count"] = self.positive_count
        return values


class DatasetSplitSummary:
    def __init__(
        self,
        training: TargetSummary,
        validation: TargetSummary,
        test: TargetSummary,
    ) -> None:
        self.training = training
        self.validation = validation
        self.test = test

    def to_dictionary(self) -> dict[str, dict[str, float | int]]:
        values: dict[str, dict[str, float | int]] = {}
        values["train"] = self.training.to_dictionary()
        values["validation"] = self.validation.to_dictionary()
        values["test"] = self.test.to_dictionary()
        return values


class DatasetDiagnostics:
    def __init__(
        self,
        row_count: int,
        data_types: dict[str, str],
        null_counts: dict[str, int],
        target_distribution: dict[str, int],
    ) -> None:
        self.row_count = row_count
        self.data_types = data_types
        self.null_counts = null_counts
        self.target_distribution = target_distribution

    def to_dictionary(self) -> dict[str, object]:
        values: dict[str, object] = {}
        values["rows"] = self.row_count
        values["dtypes"] = self.data_types
        values["nulls"] = self.null_counts
        values["target_distribution"] = self.target_distribution
        return values


def ensure_dataset_file_exists(path: Path) -> None:
    if not path.is_file():
        raise FileNotFoundError("Dataset não encontrado: " + str(path))


def read_dataset_csv(path: Path, maximum_rows: int | None) -> pd.DataFrame:
    return pd.read_csv(path, nrows=maximum_rows)


def find_missing_columns(
    frame: pd.DataFrame,
    required_columns: list[str],
) -> list[str]:
    missing_columns: list[str] = []
    for required_column in required_columns:
        if required_column not in frame.columns:
            missing_columns.append(required_column)
    missing_columns.sort()
    return missing_columns


def ensure_required_columns_exist(frame: pd.DataFrame) -> None:
    missing_columns = find_missing_columns(frame, REQUIRED_COLUMNS)
    if len(missing_columns) == 0:
        return

    found_columns = list(frame.columns)
    message = "CSV incompatível; colunas ausentes: " + str(missing_columns)
    message = message + ". Encontradas: " + str(found_columns)
    raise ValueError(message)


def ensure_dataset_is_not_empty(frame: pd.DataFrame) -> None:
    if frame.empty:
        raise ValueError("O dataset está vazio.")


def find_null_counts(frame: pd.DataFrame) -> dict[str, int]:
    counts: dict[str, int] = {}
    null_counts = frame[REQUIRED_COLUMNS].isna().sum()
    for column in REQUIRED_COLUMNS:
        count = int(null_counts[column])
        if count > 0:
            counts[column] = count
    return counts


def ensure_dataset_has_no_nulls(frame: pd.DataFrame) -> None:
    null_counts = find_null_counts(frame)
    if len(null_counts) == 0:
        return

    message = "O dataset contém valores ausentes. "
    message = message + "Política configurada: rejeitar. Detalhes: "
    message = message + str(null_counts)
    raise ValueError(message)


def ensure_target_is_binary(frame: pd.DataFrame) -> None:
    target = frame[TARGET_COLUMN]
    unique_values = target.unique().tolist()
    invalid_values: list[object] = []
    for value in unique_values:
        if value != 0:
            if value != 1:
                invalid_values.append(value)
    if len(invalid_values) > 0:
        invalid_values.sort()
        raise ValueError(
            "O alvo deve conter somente 0 e 1; encontrados: " + str(invalid_values)
        )


def ensure_target_has_both_classes(frame: pd.DataFrame) -> None:
    target = frame[TARGET_COLUMN]
    if target.nunique() != 2:
        raise ValueError("A divisão estratificada requer as duas classes no alvo.")


def select_model_columns(frame: pd.DataFrame) -> pd.DataFrame:
    return frame[REQUIRED_COLUMNS].copy()


def load_validated_dataset(path: Path, maximum_rows: int | None) -> pd.DataFrame:
    ensure_dataset_file_exists(path)
    frame = read_dataset_csv(path, maximum_rows)
    ensure_required_columns_exist(frame)
    ensure_dataset_is_not_empty(frame)
    ensure_dataset_has_no_nulls(frame)
    ensure_target_is_binary(frame)
    ensure_target_has_both_classes(frame)
    return select_model_columns(frame)


def separate_features_and_target(frame: pd.DataFrame) -> FeaturesAndTarget:
    features = frame.drop(columns=TARGET_COLUMN)
    target = frame[TARGET_COLUMN].astype("int64")
    return FeaturesAndTarget(features, target)


def calculate_relative_validation_size(config: ExperimentConfig) -> float:
    remaining_fraction = config.training_fraction + config.validation_fraction
    return config.validation_fraction / remaining_fraction


def split_test_set(
    features_and_target: FeaturesAndTarget,
    config: ExperimentConfig,
) -> RemainingAndTestData:
    split_values = train_test_split(
        features_and_target.features,
        features_and_target.target,
        test_size=config.test_fraction,
        random_state=config.seed,
        stratify=features_and_target.target,
    )
    remaining_features = split_values[0]
    test_features = split_values[1]
    remaining_target = split_values[2]
    test_target = split_values[3]
    return RemainingAndTestData(
        remaining_features,
        test_features,
        remaining_target,
        test_target,
    )


def split_training_and_validation(
    remaining_and_test: RemainingAndTestData,
    config: ExperimentConfig,
) -> DatasetSplits:
    validation_size = calculate_relative_validation_size(config)
    split_values = train_test_split(
        remaining_and_test.remaining_features,
        remaining_and_test.remaining_target,
        test_size=validation_size,
        random_state=config.seed,
        stratify=remaining_and_test.remaining_target,
    )
    training_features = split_values[0]
    validation_features = split_values[1]
    training_target = split_values[2]
    validation_target = split_values[3]

    return DatasetSplits(
        x_train=training_features,
        x_validation=validation_features,
        x_test=remaining_and_test.test_features,
        y_train=training_target,
        y_validation=validation_target,
        y_test=remaining_and_test.test_target,
    )


def create_dataset_splits(
    frame: pd.DataFrame,
    config: ExperimentConfig,
) -> DatasetSplits:
    features_and_target = separate_features_and_target(frame)
    remaining_and_test = split_test_set(features_and_target, config)
    return split_training_and_validation(remaining_and_test, config)


def summarize_target(target: pd.Series) -> TargetSummary:
    row_count = int(len(target))
    positive_rate = float(target.mean())
    positive_count = int(target.sum())
    return TargetSummary(row_count, positive_rate, positive_count)


def summarize_dataset_splits(splits: DatasetSplits) -> DatasetSplitSummary:
    training_summary = summarize_target(splits.y_train)
    validation_summary = summarize_target(splits.y_validation)
    test_summary = summarize_target(splits.y_test)
    return DatasetSplitSummary(training_summary, validation_summary, test_summary)


def create_dataset_diagnostics(frame: pd.DataFrame) -> DatasetDiagnostics:
    data_types: dict[str, str] = {}
    null_counts: dict[str, int] = {}
    target_distribution: dict[str, int] = {}

    for column in frame.columns:
        data_types[column] = str(frame[column].dtype)
        null_counts[column] = int(frame[column].isna().sum())

    distribution = frame[TARGET_COLUMN].value_counts().sort_index()
    for label in distribution.index:
        target_distribution[str(label)] = int(distribution[label])

    return DatasetDiagnostics(
        row_count=int(len(frame)),
        data_types=data_types,
        null_counts=null_counts,
        target_distribution=target_distribution,
    )



## 4. Pré-processamento

O pré-processador é ajustado somente no treino; validação e teste apenas transformam.

In [ ]:
"""Pré-processamento e criação explícita dos DataLoaders binários."""

import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset


class TransformedSplits:
    def __init__(self, training: np.ndarray, validation: np.ndarray, test: np.ndarray) -> None:
        self.training = training
        self.validation = validation
        self.test = test


class DataLoaders:
    def __init__(self, train: DataLoader, validation: DataLoader, test: DataLoader) -> None:
        self.train = train
        self.validation = validation
        self.test = test


def create_categorical_transformer() -> OneHotEncoder:
    return OneHotEncoder(handle_unknown="ignore", sparse_output=False)


def create_numeric_transformer() -> Pipeline:
    steps = [("scaler", StandardScaler())]
    return Pipeline(steps)


def create_preprocessor() -> ColumnTransformer:
    categorical_transformer = create_categorical_transformer()
    numeric_transformer = create_numeric_transformer()
    transformers = [
        ("categorical", categorical_transformer, CATEGORICAL_COLUMNS),
        ("numeric", numeric_transformer, NUMERIC_COLUMNS),
    ]
    return ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.0)


def convert_matrix_to_float32(matrix: object) -> np.ndarray:
    return np.asarray(matrix, dtype=np.float32)


def fit_and_transform_training_data(preprocessor: ColumnTransformer, training_features: pd.DataFrame) -> np.ndarray:
    return convert_matrix_to_float32(preprocessor.fit_transform(training_features))


def transform_validation_data(preprocessor: ColumnTransformer, validation_features: pd.DataFrame) -> np.ndarray:
    return convert_matrix_to_float32(preprocessor.transform(validation_features))


def transform_test_data(preprocessor: ColumnTransformer, test_features: pd.DataFrame) -> np.ndarray:
    return convert_matrix_to_float32(preprocessor.transform(test_features))


def ensure_equal_feature_widths(transformed_splits: TransformedSplits) -> None:
    training_width = transformed_splits.training.shape[1]
    validation_width = transformed_splits.validation.shape[1]
    test_width = transformed_splits.test.shape[1]
    if training_width != validation_width or training_width != test_width:
        widths: dict[str, int] = {}
        widths["train"] = training_width
        widths["validation"] = validation_width
        widths["test"] = test_width
        raise RuntimeError("Dimensões transformadas incompatíveis: " + str(widths))


def transform_dataset_splits(splits: DatasetSplits, preprocessor: ColumnTransformer) -> TransformedSplits:
    training = fit_and_transform_training_data(preprocessor, splits.x_train)
    validation = transform_validation_data(preprocessor, splits.x_validation)
    test = transform_test_data(preprocessor, splits.x_test)
    transformed_splits = TransformedSplits(training, validation, test)
    ensure_equal_feature_widths(transformed_splits)
    return transformed_splits


def convert_labels_to_float32_column(target: pd.Series) -> np.ndarray:
    values = target.to_numpy(dtype=np.float32, copy=True)
    return values.reshape(-1, 1)


def create_tensor_dataset(features: np.ndarray, labels: np.ndarray) -> TensorDataset:
    feature_tensor = torch.from_numpy(features)
    label_tensor = torch.from_numpy(labels)
    return TensorDataset(feature_tensor, label_tensor)


def create_training_generator(seed: int) -> torch.Generator:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


def create_data_loader(dataset: TensorDataset, batch_size: int, shuffle: bool, pin_memory: bool, generator: torch.Generator | None) -> DataLoader:
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, pin_memory=pin_memory, generator=generator, num_workers=0)


def create_data_loaders(transformed_splits: TransformedSplits, splits: DatasetSplits, config: ExperimentConfig, device: torch.device) -> DataLoaders:
    training_labels = convert_labels_to_float32_column(splits.y_train)
    validation_labels = convert_labels_to_float32_column(splits.y_validation)
    test_labels = convert_labels_to_float32_column(splits.y_test)
    training_dataset = create_tensor_dataset(transformed_splits.training, training_labels)
    validation_dataset = create_tensor_dataset(transformed_splits.validation, validation_labels)
    test_dataset = create_tensor_dataset(transformed_splits.test, test_labels)
    pin_memory = device.type == "cuda"
    training_generator = create_training_generator(config.seed)
    training_loader = create_data_loader(training_dataset, config.batch_size, True, pin_memory, training_generator)
    validation_loader = create_data_loader(validation_dataset, config.batch_size, False, pin_memory, None)
    test_loader = create_data_loader(test_dataset, config.batch_size, False, pin_memory, None)
    return DataLoaders(training_loader, validation_loader, test_loader)


def get_transformed_feature_count(transformed_splits: TransformedSplits) -> int:
    return int(transformed_splits.training.shape[1])


## 5. Modelo

A MLP é montada em blocos Linear, BatchNorm, ReLU e Dropout.

In [ ]:
"""Definição explícita da MLP binária de saída única."""

import torch
from torch import nn


def validate_hidden_dimensions(hidden_dimensions: list[int]) -> None:
    if len(hidden_dimensions) == 0:
        raise ValueError("A MLP precisa de ao menos uma dimensão oculta.")
    for hidden_dimension in hidden_dimensions:
        if hidden_dimension <= 0:
            raise ValueError("As dimensões ocultas devem ser positivas.")


def create_hidden_block(input_size: int, output_size: int, dropout: float) -> list[nn.Module]:
    modules: list[nn.Module] = []
    modules.append(nn.Linear(input_size, output_size))
    modules.append(nn.BatchNorm1d(output_size))
    modules.append(nn.ReLU())
    modules.append(nn.Dropout(dropout))
    return modules


def create_network_layers(input_size: int, hidden_dimensions: list[int], output_size: int, dropout: float) -> list[nn.Module]:
    validate_hidden_dimensions(hidden_dimensions)
    if output_size != 1:
        raise ValueError("A MLP binária precisa ter uma única saída.")
    layers: list[nn.Module] = []
    previous_size = input_size
    for hidden_dimension in hidden_dimensions:
        hidden_block = create_hidden_block(previous_size, hidden_dimension, dropout)
        for module in hidden_block:
            layers.append(module)
        previous_size = hidden_dimension
    layers.append(nn.Linear(previous_size, output_size))
    return layers


class MLP(nn.Module):
    def __init__(self, input_size: int, hidden_dimensions: list[int], output_size: int, dropout: float) -> None:
        super().__init__()
        self.network = nn.Sequential(*create_network_layers(input_size, hidden_dimensions, output_size, dropout))
        self.input_size = input_size
        self.hidden_dimensions = list(hidden_dimensions)
        self.output_size = output_size
        self.dropout = dropout

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.network(features)


def create_model(input_size: int, config: ExperimentConfig, device: torch.device) -> MLP:
    model = MLP(input_size, config.hidden_dimensions, config.output_size, config.dropout)
    model.to(device)
    return model


def validate_model_output(model: MLP, sample_features: torch.Tensor, output_size: int, device: torch.device) -> None:
    model.eval()
    features_on_device = sample_features.to(device)
    with torch.no_grad():
        logits = model(features_on_device)
    expected_shape = (sample_features.shape[0], output_size)
    if logits.shape != expected_shape:
        message = "Formato de saída inválido. Esperado: " + str(expected_shape)
        message = message + ". Recebido: " + str(tuple(logits.shape))
        raise RuntimeError(message)
    if not torch.isfinite(logits).all():
        raise RuntimeError("A saída do modelo contém valores não finitos.")


## 6. Treinamento

Lote, época, checkpoint e early stopping são responsabilidades separadas.

In [ ]:
"""Treino binário, F1 positivo, calibração de limiar e checkpoint."""
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch import nn
from torch.utils.data import DataLoader

class DeviceBatch:
    def __init__(self, features, labels): self.features, self.labels = features, labels
class BatchTrainingResult:
    def __init__(self, weighted_loss, example_count): self.weighted_loss, self.example_count = weighted_loss, example_count
class BatchPrediction:
    def __init__(self, weighted_loss, example_count, actual_labels, positive_probabilities): self.weighted_loss, self.example_count, self.actual_labels, self.positive_probabilities = weighted_loss, example_count, actual_labels, positive_probabilities
class EpochResult:
    def __init__(self, average_loss, actual_labels, positive_probabilities): self.average_loss, self.actual_labels, self.positive_probabilities = average_loss, actual_labels, positive_probabilities
class PositiveClassMetrics:
    def __init__(self, accuracy, precision, recall, f1): self.accuracy, self.precision, self.recall, self.f1 = accuracy, precision, recall, f1
    def to_dictionary(self): return {"accuracy": self.accuracy, "precision_positive": self.precision, "recall_positive": self.recall, "f1_positive": self.f1}
class ThresholdSelection:
    def __init__(self, threshold, metrics): self.threshold, self.metrics = threshold, metrics
    def to_dictionary(self): return {**self.metrics.to_dictionary(), "positive_threshold": self.threshold}
class TrainingHistory:
    def __init__(self): self.training_losses, self.validation_losses, self.validation_accuracies, self.validation_precisions, self.validation_recalls, self.validation_f1_scores, self.validation_thresholds = [], [], [], [], [], [], []
    def add_epoch(self, training_loss, validation_loss, selection):
        self.training_losses.append(training_loss); self.validation_losses.append(validation_loss); self.validation_accuracies.append(selection.metrics.accuracy); self.validation_precisions.append(selection.metrics.precision); self.validation_recalls.append(selection.metrics.recall); self.validation_f1_scores.append(selection.metrics.f1); self.validation_thresholds.append(selection.threshold)
    def to_dictionary(self): return {"train_loss": self.training_losses, "val_loss": self.validation_losses, "val_acc": self.validation_accuracies, "val_precision": self.validation_precisions, "val_recall": self.validation_recalls, "val_positive_f1": self.validation_f1_scores, "val_threshold": self.validation_thresholds}
class TrainingOutcome:
    def __init__(self, history, validation_selection): self.history, self.validation_selection = history, validation_selection

def move_batch_to_device(batch, device): return DeviceBatch(batch[0].to(device, non_blocking=device.type == "cuda"), batch[1].to(device, non_blocking=device.type == "cuda"))
def train_single_batch(model, batch, loss_function, optimizer, device):
    item=move_batch_to_device(batch, device); optimizer.zero_grad(set_to_none=True); loss=loss_function(model(item.features), item.labels); loss.backward(); optimizer.step(); return BatchTrainingResult(float(loss.item()) * int(item.labels.size(0)), int(item.labels.size(0)))
def train_one_epoch(model, loader, loss_function, optimizer, device):
    model.train(); total_loss=0.; total=0
    for batch in loader: result=train_single_batch(model,batch,loss_function,optimizer,device); total_loss+=result.weighted_loss; total+=result.example_count
    if not total: raise RuntimeError("Não é possível treinar com um DataLoader vazio.")
    return EpochResult(total_loss/total,None,None)
def convert_logits_to_positive_probabilities(logits): return torch.sigmoid(logits).reshape(-1)
def predict_single_batch(model,batch,loss_function,device):
    item=move_batch_to_device(batch,device); logits=model(item.features); loss=loss_function(logits,item.labels); count=int(item.labels.size(0)); return BatchPrediction(float(loss.item())*count,count,item.labels.reshape(-1).to(torch.int64).cpu().tolist(),convert_logits_to_positive_probabilities(logits).cpu().tolist())
def evaluate_one_epoch(model,loader,loss_function,device):
    model.eval(); total_loss=0.; total=0; labels=[]; probabilities=[]
    with torch.no_grad():
        for batch in loader: result=predict_single_batch(model,batch,loss_function,device); total_loss+=result.weighted_loss; total+=result.example_count; labels.extend(result.actual_labels); probabilities.extend(result.positive_probabilities)
    if not total: raise RuntimeError("Não é possível avaliar com um DataLoader vazio.")
    return EpochResult(total_loss/total,np.asarray(labels,dtype=np.int64),np.asarray(probabilities,dtype=np.float32))
def create_predictions_from_threshold(probabilities,threshold): return (probabilities >= threshold).astype(np.int64)
def calculate_positive_class_metrics(actual,predicted): return PositiveClassMetrics(float(accuracy_score(actual,predicted)),float(precision_score(actual,predicted,pos_label=1,zero_division=0)),float(recall_score(actual,predicted,pos_label=1,zero_division=0)),float(f1_score(actual,predicted,pos_label=1,zero_division=0)))
def create_threshold_values(config): return [round(config.threshold_minimum + index * config.threshold_step, 10) for index in range(round((config.threshold_maximum-config.threshold_minimum)/config.threshold_step)+1)]
def is_better_threshold(candidate,current): return current is None or candidate.metrics.f1 > current.metrics.f1 + 1e-12 or (abs(candidate.metrics.f1-current.metrics.f1)<=1e-12 and abs(candidate.threshold-.5)<abs(current.threshold-.5))
def select_best_threshold(result,config):
    if result.actual_labels is None or result.positive_probabilities is None: raise ValueError("A seleção de limiar exige rótulos e probabilidades.")
    best=None
    for threshold in create_threshold_values(config):
        candidate=ThresholdSelection(threshold,calculate_positive_class_metrics(result.actual_labels,create_predictions_from_threshold(result.positive_probabilities,threshold)))
        if is_better_threshold(candidate,best): best=candidate
    return best
def calculate_positive_class_weight(training_target,multiplier):
    if multiplier == 0: return None
    positives=int((training_target.to_numpy(dtype=np.int64)==1).sum()); negatives=len(training_target)-positives
    if not positives or not negatives: raise ValueError("O treino precisa conter exemplos das duas classes.")
    return torch.tensor([multiplier * negatives / positives],dtype=torch.float32)
def create_loss_function(weight,device): return nn.BCEWithLogitsLoss(pos_weight=None if weight is None else weight.to(device))
def create_optimizer(model,config): return torch.optim.AdamW(model.parameters(),lr=config.learning_rate,weight_decay=config.weight_decay)
def validation_positive_f1_improved(current,best): return current > best + 1e-12
def should_stop_early(epoch,without_improvement,minimum_epochs,patience): return epoch >= minimum_epochs and without_improvement >= patience
def create_checkpoint_data(model,epoch,selection,history,config): return {"state_dict":model.state_dict(),"in_dim":model.input_size,"hidden_dims":list(model.hidden_dimensions),"output_size":model.output_size,"dropout":model.dropout,"epoch":epoch,"positive_threshold":selection.threshold,"validation_selection":selection.to_dictionary(),"history":history.to_dictionary(),"config":config_to_dictionary(config)}
def save_best_checkpoint(checkpoint,path): path.parent.mkdir(parents=True,exist_ok=True); torch.save(checkpoint,path)
def create_selection_from_dictionary(values): return ThresholdSelection(float(values["positive_threshold"]),PositiveClassMetrics(float(values["accuracy"]),float(values["precision_positive"]),float(values["recall_positive"]),float(values["f1_positive"])))
def restore_best_model(model,path,device):
    checkpoint=torch.load(path,map_location=device,weights_only=False)
    if int(checkpoint["output_size"]) != 1: raise ValueError("O checkpoint não usa o contrato de saída única.")
    model.load_state_dict(checkpoint["state_dict"]); model.to(device); return checkpoint
def print_epoch_summary(epoch,train_loss,val_loss,selection): print(f"epoch={epoch:03d} train_loss={train_loss:.5f} val_loss={val_loss:.5f} threshold={selection.threshold:.2f} val_recall={selection.metrics.recall:.4f} val_positive_f1={selection.metrics.f1:.4f}")
def train_model(model,train_loader,val_loader,loss,optimizer,config,device,checkpoint_path):
    history=TrainingHistory(); best=float("-inf"); without=0
    for epoch in range(1,config.epochs+1):
        train=train_one_epoch(model,train_loader,loss,optimizer,device); val=evaluate_one_epoch(model,val_loader,loss,device); selection=select_best_threshold(val,config); history.add_epoch(train.average_loss,val.average_loss,selection)
        if epoch % config.log_interval == 0: print_epoch_summary(epoch,train.average_loss,val.average_loss,selection)
        if validation_positive_f1_improved(selection.metrics.f1,best): best=selection.metrics.f1; without=0; save_best_checkpoint(create_checkpoint_data(model,epoch,selection,history,config),checkpoint_path)
        else: without+=1
        if should_stop_early(epoch,without,config.minimum_epochs,config.early_stopping_patience): print(f"Early stopping na época {epoch}; paciência={config.early_stopping_patience}."); break
    checkpoint=restore_best_model(model,checkpoint_path,device); return TrainingOutcome(history,create_selection_from_dictionary(checkpoint["validation_selection"]))

## 7. Avaliação

As métricas são calculadas a partir de um único percurso pelo DataLoader.

In [ ]:
"""Cálculo das métricas finais no limiar calibrado."""
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
class ClassificationMetrics:
    def __init__(self,loss,positive_threshold,positive_metrics,report_dictionary,report_text,confusion): self.loss,self.positive_threshold,self.positive_metrics,self.report_dictionary,self.report_text,self.confusion=loss,positive_threshold,positive_metrics,report_dictionary,report_text,confusion
    def to_dictionary(self): return {"loss":self.loss,"positive_threshold":self.positive_threshold,**self.positive_metrics.to_dictionary(),"classification_report":self.report_dictionary}
def calculate_classification_metrics(result,threshold):
    if result.actual_labels is None or result.positive_probabilities is None: raise ValueError("O resultado não contém dados de avaliação.")
    predicted=create_predictions_from_threshold(result.positive_probabilities,threshold)
    return ClassificationMetrics(result.average_loss,threshold,calculate_positive_class_metrics(result.actual_labels,predicted),classification_report(result.actual_labels,predicted,output_dict=True,zero_division=0),classification_report(result.actual_labels,predicted,zero_division=0),confusion_matrix(result.actual_labels,predicted,labels=[0,1]))
def evaluate_test_set(model,loader,loss_function,device,positive_threshold): return calculate_classification_metrics(evaluate_one_epoch(model,loader,loss_function,device),positive_threshold)

## 8. Artefatos

Cada arquivo ou gráfico possui uma operação de persistência nomeada.

In [ ]:
"""Persistência dos artefatos do experimento calibrado."""
import json
from pathlib import Path
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer

def ensure_artifacts_directory_exists(directory): directory.mkdir(parents=True,exist_ok=True)
def save_json_file(path,values): path.write_text(json.dumps(values,ensure_ascii=False,indent=2),encoding="utf-8")
def save_text_file(path,value): path.write_text(value,encoding="utf-8")
def save_preprocessor(path,preprocessor): joblib.dump(preprocessor,path)
def load_preprocessor(path): return joblib.load(path)
def save_test_metrics(metrics,directory): save_json_file(directory / "test_metrics.json",metrics.to_dictionary())
def save_threshold_selection(selection,directory): save_json_file(directory / "threshold_selection.json",selection.to_dictionary())
def save_classification_report(metrics,directory): save_text_file(directory / "classification_report.txt",metrics.report_text)
def save_confusion_matrix_figure(metrics,directory):
    figure,axis=plt.subplots(figsize=(5,4)); sns.heatmap(metrics.confusion,annot=True,fmt="d",cmap="Blues",xticklabels=["Negativa","Positiva"],yticklabels=["Negativa","Positiva"],ax=axis); axis.set_xlabel("Predito"); axis.set_ylabel("Real"); figure.tight_layout(); figure.savefig(directory / "confusion_matrix.png",dpi=160); plt.close(figure)
def save_learning_curves_figure(history,directory):
    epochs=range(1,len(history.training_losses)+1); figure,axes=plt.subplots(1,2,figsize=(11,4)); axes[0].plot(epochs,history.training_losses,label="Treino"); axes[0].plot(epochs,history.validation_losses,label="Validação"); axes[0].set_title("Perda"); axes[1].plot(epochs,history.validation_recalls,label="Recall positivo"); axes[1].plot(epochs,history.validation_f1_scores,label="F1 positivo"); axes[1].set_title("Métricas da classe positiva")
    for axis in axes: axis.grid(True,alpha=.3); axis.legend()
    figure.tight_layout(); figure.savefig(directory / "learning_curves.png",dpi=160); plt.close(figure)
def save_history(history,directory): save_json_file(directory / "history.json",history.to_dictionary())
def save_metadata(metadata,directory): save_json_file(directory / "metadata.json",metadata)
def save_imbalance_report(metrics,directory): save_text_file(directory / "imbalance_report.txt",f"Limiar positivo: {metrics.positive_threshold:.2f}\nAcurácia: {metrics.positive_metrics.accuracy:.5f}\nPrecision positiva: {metrics.positive_metrics.precision:.5f}\nRecall positivo: {metrics.positive_metrics.recall:.5f}\nF1 positivo: {metrics.positive_metrics.f1:.5f}\nConclusão: F1 positivo orienta a detecção da classe minoritária.\n")
def save_experiment_artifacts(preprocessor,preprocessor_path,metrics,history,selection,directory):
    ensure_artifacts_directory_exists(directory); save_preprocessor(preprocessor_path,preprocessor); save_test_metrics(metrics,directory); save_threshold_selection(selection,directory); save_classification_report(metrics,directory); save_confusion_matrix_figure(metrics,directory); save_learning_curves_figure(history,directory); save_history(history,directory); save_imbalance_report(metrics,directory)

## 9. Inferência

Modelo e pré-processador são carregados uma vez e reutilizados.

In [ ]:
"""Inferência binária com sigmoid e limiar salvo."""

from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer


class LoadedModel:
    def __init__(self, model: MLP, positive_threshold: float) -> None:
        self.model = model
        self.positive_threshold = positive_threshold


class Predictor:
    def __init__(self, model: MLP, preprocessor: ColumnTransformer, device: torch.device, positive_threshold: float) -> None:
        self.model = model
        self.preprocessor = preprocessor
        self.device = device
        self.positive_threshold = positive_threshold


def ensure_inference_columns_exist(frame: pd.DataFrame) -> None:
    missing_columns = find_missing_columns(frame, FEATURE_COLUMNS)
    if len(missing_columns) > 0:
        raise ValueError("Entradas sem colunas exigidas: " + str(missing_columns))


def select_inference_columns(frame: pd.DataFrame) -> pd.DataFrame:
    return frame[FEATURE_COLUMNS]


def load_model_from_checkpoint(path: Path, device: torch.device) -> LoadedModel:
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    if "output_size" not in checkpoint:
        raise ValueError("Checkpoint antigo de duas saídas não é compatível.")
    output_size = int(checkpoint["output_size"])
    if output_size != 1:
        raise ValueError("O checkpoint não usa uma saída binária única.")
    model = MLP(int(checkpoint["in_dim"]), list(checkpoint["hidden_dims"]), output_size, float(checkpoint["dropout"]))
    model.to(device)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    threshold = float(checkpoint["positive_threshold"])
    return LoadedModel(model, threshold)


def transform_inference_features(frame: pd.DataFrame, preprocessor: ColumnTransformer) -> np.ndarray:
    transformed = preprocessor.transform(select_inference_columns(frame))
    return np.asarray(transformed, dtype=np.float32)


def predict_classes(model: MLP, transformed_features: np.ndarray, device: torch.device, positive_threshold: float) -> np.ndarray:
    features = torch.from_numpy(transformed_features).to(device)
    with torch.no_grad():
        logits = model(features)
        probabilities = convert_logits_to_positive_probabilities(logits)
    probability_values = probabilities.cpu().numpy()
    return create_predictions_from_threshold(probability_values, positive_threshold)


def load_predictor(checkpoint_path: Path, preprocessor_path: Path, device: torch.device) -> Predictor:
    loaded_model = load_model_from_checkpoint(checkpoint_path, device)
    preprocessor = load_preprocessor(preprocessor_path)
    return Predictor(loaded_model.model, preprocessor, device, loaded_model.positive_threshold)


def predict(predictor: Predictor, frame: pd.DataFrame) -> np.ndarray:
    ensure_inference_columns_exist(frame)
    features = transform_inference_features(frame, predictor.preprocessor)
    return predict_classes(predictor.model, features, predictor.device, predictor.positive_threshold)


## 10. Orquestração

O fluxo principal apenas coordena as funções definidas anteriormente.

In [ ]:
"""Orquestração do experimento e estudo reproduzível de pesos."""
import json
import shutil
import copy
from dataclasses import dataclass
from pathlib import Path
from torch import nn

class PreparedExperiment:
    def __init__(self,config,device,runtime_metadata): self.config,self.device,self.runtime_metadata=config,device,runtime_metadata
class PreparedTrainingData:
    def __init__(self,frame,splits,split_summary,diagnostics,preprocessor,loaders,input_size): self.frame,self.splits,self.split_summary,self.diagnostics,self.preprocessor,self.loaders,self.input_size=frame,splits,split_summary,diagnostics,preprocessor,loaders,input_size
class TrainingComponents:
    def __init__(self,model,loss_function,optimizer,positive_class_weight): self.model,self.loss_function,self.optimizer,self.positive_class_weight=model,loss_function,optimizer,positive_class_weight
class ExperimentResult:
    def __init__(self,config,runtime_metadata,diagnostics,split_summary,training_outcome,test_metrics,inference_predictions,input_size,positive_class_weight): self.config,self.runtime_metadata,self.diagnostics,self.split_summary,self.training_outcome,self.test_metrics,self.inference_predictions,self.input_size,self.positive_class_weight=config,runtime_metadata,diagnostics,split_summary,training_outcome,test_metrics,inference_predictions,input_size,positive_class_weight
    def to_dictionary(self): return {**self.runtime_metadata.to_dictionary(),"config":config_to_dictionary(self.config),"diagnostics":self.diagnostics.to_dictionary(),"splits":self.split_summary.to_dictionary(),"history":self.training_outcome.history.to_dictionary(),"validation_threshold_selection":self.training_outcome.validation_selection.to_dictionary(),"test_metrics":self.test_metrics.to_dictionary(),"inference_smoke_predictions":self.inference_predictions,"positive_class_weight":self.positive_class_weight,"in_dim":self.input_size,"selection_metric":"positive_f1","selection_reason":"Maior F1 positivo de validação com limiar calibrado."}
@dataclass(frozen=True)
class CandidateScore:
    multiplier: float
    val_positive_f1: float
    val_loss: float
    best_threshold: float
    selected_epoch: int
    artifacts_subdir: str
    def to_dictionary(self): return self.__dict__
def prepare_experiment(config): validate_config(config); configure_reproducibility(config.seed); device=select_device(config.requested_device); metadata=collect_runtime_metadata(device); print_device_summary(metadata); ensure_artifacts_directory_exists(config.artifacts_directory); return PreparedExperiment(config,device,metadata)
def prepare_training_data(config,device):
    frame=load_validated_dataset(config.data_path,config.maximum_rows); diagnostics=create_dataset_diagnostics(frame); splits=create_dataset_splits(frame,config); summary=summarize_dataset_splits(splits); preprocessor=create_preprocessor(); transformed=transform_dataset_splits(splits,preprocessor); return PreparedTrainingData(frame,splits,summary,diagnostics,preprocessor,create_data_loaders(transformed,splits,config,device),get_transformed_feature_count(transformed))
def prepare_training_components(data,config,device):
    model=create_model(data.input_size,config,device); validate_model_output(model,next(iter(data.loaders.train))[0],config.output_size,device); weight=calculate_positive_class_weight(data.splits.y_train,config.positive_weight_multiplier); return TrainingComponents(model,create_loss_function(weight,device),create_optimizer(model,config),None if weight is None else float(weight.item()))
def run_experiment(config):
    prepared=prepare_experiment(config); data=prepare_training_data(config,prepared.device); components=prepare_training_components(data,config,prepared.device); outcome=train_model(components.model,data.loaders.train,data.loaders.validation,components.loss_function,components.optimizer,config,prepared.device,get_checkpoint_path(config)); metrics=evaluate_test_set(components.model,data.loaders.test,components.loss_function,prepared.device,outcome.validation_selection.threshold); save_experiment_artifacts(data.preprocessor,get_preprocessor_path(config),metrics,outcome.history,outcome.validation_selection,config.artifacts_directory); predictor=load_predictor(get_checkpoint_path(config),get_preprocessor_path(config),prepared.device); result=ExperimentResult(config,prepared.runtime_metadata,data.diagnostics,data.split_summary,outcome,metrics,predict(predictor,data.splits.x_validation.head(3)).tolist(),data.input_size,components.positive_class_weight); metadata=result.to_dictionary(); metadata.update({"threshold_minimum":config.threshold_minimum,"threshold_maximum":config.threshold_maximum,"threshold_step":config.threshold_step}); save_metadata(metadata,config.artifacts_directory); return result
def run_weight_study(config_template,multipliers=(0.0,.25,.50,.75,1.0),study_directory=None,final_directory=None):
    candidates=[]; results=[]
    for multiplier in multipliers:
        config=copy.deepcopy(config_template); config.positive_weight_multiplier=multiplier; config.artifacts_directory=config.project_root / "artifacts" / "_studies" / f"weight_{multiplier:.2f}"; result=run_experiment(config); checkpoint=torch.load(get_checkpoint_path(config),map_location="cpu",weights_only=False); candidates.append(CandidateScore(multiplier,result.training_outcome.validation_selection.metrics.f1,result.training_outcome.history.validation_losses[checkpoint["epoch"]-1],result.training_outcome.validation_selection.threshold,checkpoint["epoch"],str(config.artifacts_directory))); results.append(result)
    ordered=sorted(enumerate(candidates),key=lambda item:(-item[1].val_positive_f1,item[1].val_loss,item[0])); winner_index,winner=ordered[0]; final_directory=config_template.project_root / "artifacts" / "f1_full_dataset"; shutil.copytree(Path(winner.artifacts_subdir),final_directory,dirs_exist_ok=True); summary={"selection_metric":"positive_f1","candidates":[candidate.to_dictionary() for candidate in candidates],"winner_multiplier":winner.multiplier,"winner_artifacts_subdir":winner.artifacts_subdir,"test_evaluated_candidates":[winner.multiplier]}; save_json_file(final_directory / "weight_study_summary.json",summary); return results[winner_index],summary
def verify_reproducibility(project_root,maximum_rows,epochs):
    first=create_default_config(project_root); first.requested_device="cpu"; first.maximum_rows=maximum_rows; first.epochs=epochs; first.artifacts_directory=project_root / "artifacts" / "_studies" / "reproducibility" / "run_1"; second=copy.deepcopy(first); second.artifacts_directory=project_root / "artifacts" / "_studies" / "reproducibility" / "run_2"; one,two=run_experiment(first),run_experiment(second)
    if one.to_dictionary()["test_metrics"] != two.to_dictionary()["test_metrics"]: raise AssertionError("Execuções CPU com a mesma semente divergiram.")
    return {"device":"cpu","result":"identical deterministic CPU runs"}
def verify_cuda(project_root,maximum_rows,epochs):
    if not torch.cuda.is_available(): return {"device":"cuda","available":False,"message":"CUDA não está disponível neste ambiente."}
    config=create_default_config(project_root); config.requested_device="cuda"; config.maximum_rows=maximum_rows; config.epochs=epochs; directory=project_root / "artifacts" / "_studies" / "cuda_validation"; return run_weight_study(config, study_directory=directory, final_directory=directory / "final")

In [ ]:
def run_weight_candidate(config):
    prepared=prepare_experiment(config)
    data=prepare_training_data(config,prepared.device)
    components=prepare_training_components(data,config,prepared.device)
    outcome=train_model(components.model,data.loaders.train,data.loaders.validation,components.loss_function,components.optimizer,config,prepared.device,get_checkpoint_path(config))
    save_preprocessor(get_preprocessor_path(config),data.preprocessor)
    save_history(outcome.history,config.artifacts_directory)
    save_threshold_selection(outcome.validation_selection,config.artifacts_directory)
    checkpoint=torch.load(get_checkpoint_path(config),map_location="cpu",weights_only=False)
    return CandidateScore(config.positive_weight_multiplier,outcome.validation_selection.metrics.f1,outcome.history.validation_losses[checkpoint["epoch"]-1],outcome.validation_selection.threshold,checkpoint["epoch"],str(config.artifacts_directory))


def run_weight_study(config_template,multipliers=(0.0,.25,.50,.75,1.0),study_directory=None,final_directory=None):
    candidates=[]
    for multiplier in multipliers:
        candidate_config=copy.deepcopy(config_template)
        candidate_config.positive_weight_multiplier=multiplier
        root=study_directory or candidate_config.project_root / "artifacts" / "_studies"
        candidate_config.artifacts_directory=root / f"weight_{multiplier:.2f}"
        candidates.append(run_weight_candidate(candidate_config))
    winner_index,winner=sorted(enumerate(candidates),key=lambda item:(-item[1].val_positive_f1,item[1].val_loss,item[0]))[0]
    final_config=copy.deepcopy(config_template)
    final_config.positive_weight_multiplier=winner.multiplier
    final_config.artifacts_directory=final_directory or final_config.project_root / "artifacts" / "f1_full_dataset"
    winner_result=run_experiment(final_config)
    summary={"selection_metric":"positive_f1","candidates":[candidate.to_dictionary() for candidate in candidates],"winner_multiplier":winner.multiplier,"winner_artifacts_subdir":winner.artifacts_subdir,"test_evaluated_candidates":[winner.multiplier]}
    save_json_file(final_config.artifacts_directory / "weight_study_summary.json",summary)
    return winner_result,summary

In [ ]:
def run_synthetic_checks() -> None:
    config=create_default_config(Path.cwd())
    if create_threshold_values(config) != [round(.05+i*.01,10) for i in range(91)]: raise AssertionError("Faixa de limiares incorreta.")
    labels=np.asarray([1,1,0,0]); probabilities=np.asarray([.4,.8,.3,.1]); selection=select_best_threshold(EpochResult(0.,labels,probabilities),config)
    if selection.metrics.f1 != 1.0: raise AssertionError("A seleção não priorizou F1 positivo.")
    tie=select_best_threshold(EpochResult(0.,np.asarray([1,1,0]),np.asarray([.6,.55,.2])),config)
    if tie.threshold != .5: raise AssertionError("O desempate não priorizou 0.50.")
    target=pd.Series([0,0,0,1])
    if calculate_positive_class_weight(target,0) is not None: raise AssertionError("Multiplicador zero deveria desativar pesos.")
    if abs(float(calculate_positive_class_weight(target,.5).item())-1.5)>1e-12: raise AssertionError("Multiplicador de peso incorreto.")
    if should_stop_early(29,8,30,8) or not should_stop_early(30,8,30,8): raise AssertionError("Early stopping incorreto.")
    print("Verificações sintéticas aprovadas.")
run_synthetic_checks()

## 12. Estratégia de melhoria: F1 positivo, limiar e pesos

A MLP usa Entrada → 32 → 16 → 1. A única saída é um logit: sigmoid a converte em probabilidade positiva apenas para validação, teste e inferência. BCEWithLogitsLoss recebe pos_weight, calculado somente com o treino e controlado por positive_weight_multiplier. O maior F1 positivo de validação escolhe checkpoint e limiar; o teste permanece isolado até a escolha do vencedor.


## 13. Configurar a execução

O treino final usa o dataset completo, a arquitetura `Entrada → 32 → 16 → 1`, no mínimo 30 e no máximo 200 épocas.

In [ ]:
PROJECT_ROOT=Path.cwd()
config=create_default_config(PROJECT_ROOT)
config.maximum_rows=None
config.epochs=200
config.minimum_epochs=30
config.positive_weight_multiplier=1.0
config.artifacts_directory=PROJECT_ROOT / "artifacts" / "f1_full_dataset"
validate_config(config)
config_to_dictionary(config)

## 14. Executar o experimento

A chamada prepara os dados, calibra o limiar na validação, treina, avalia uma única vez no teste e salva os artefatos.

In [ ]:
winner_result, weight_study_summary = run_weight_study(config)
winner_result.to_dictionary()

## 15. Validações opcionais

As chamadas permanecem comentadas porque treinam novamente.

In [ ]:
# reproducibility_result = verify_reproducibility(PROJECT_ROOT, maximum_rows=3000, epochs=30)
# cuda_result = verify_cuda(PROJECT_ROOT, maximum_rows=3000, epochs=30)